# Download NOAA water-level observations

Retrieve public six-minute water-level observations using the [NOAA CO-OPS Data API](https://api.tidesandcurrents.noaa.gov/api/prod/).

Run the cells in order after installing `requests` in your notebook's Python environment (see the accompanying README). Requests are split into conservative 28-day chunks to stay within NOAA's one-month limit. Dates are inclusive; single-day requests are supported.

The example retains the original station ID and date range but removes inconsistent place names. Confirm station and datum availability using [NOAA's station directory](https://tidesandcurrents.noaa.gov/stations.html). Values are in **meters**, timestamps are **GMT**, and **MSL means Mean Sea Level**.

HTTP errors, malformed responses, and empty chunks stop the download before a CSV is written. Existing files are never overwritten. NOAA quality columns and missing values are preserved; successful retrieval does not establish scientific validity or temporal completeness. Review gaps and flags before analysis. No credentials are required.


In [ ]:
import csv
import io
from datetime import datetime, timedelta
from pathlib import Path

import requests

API_URL = "https://api.tidesandcurrents.noaa.gov/api/prod/datagetter"


def date_ranges(start_date, end_date):
    """Yield inclusive ranges of at most 28 days, including single-day requests."""
    start = datetime.strptime(start_date, "%Y%m%d").date()
    end = datetime.strptime(end_date, "%Y%m%d").date()
    if start > end:
        raise ValueError("start_date must be on or before end_date")
    while start <= end:
        stop = min(start + timedelta(days=27), end)
        yield start.strftime("%Y%m%d"), stop.strftime("%Y%m%d")
        start = stop + timedelta(days=1)


def download_water_levels(station, start_date, end_date, datum="MSL"):
    """Download all chunks or raise; retain NOAA values and quality columns as text.

    Water levels use metric units (meters), with GMT timestamps.
    A successful response does not guarantee a complete or quality-controlled series.
    """
    if not isinstance(station, str) or len(station) != 7 or not station.isascii() or not station.isdigit():
        raise ValueError("station must be a seven-digit string")
    allowed_datums = {"CRD", "IGLD", "LWD", "MHHW", "MHW", "MTL",
                      "MSL", "MLW", "MLLW", "NAVD", "STND"}
    if datum not in allowed_datums:
        raise ValueError("Unsupported datum; confirm availability for your station")
    ranges = list(date_ranges(start_date, end_date))
    rows_by_time = {}
    columns = None
    with requests.Session() as session:
        for begin, end in ranges:
            params = {
                "station": station, "begin_date": begin, "end_date": end,
                "product": "water_level", "datum": datum,
                "units": "metric", "time_zone": "gmt",
                "application": "research_python_example", "format": "csv",
            }
            response = session.get(API_URL, params=params, timeout=(10, 60))
            response.raise_for_status()
            reader = csv.DictReader(io.StringIO(response.text.strip()))
            headers = [name.strip() for name in (reader.fieldnames or [])]
            if not {"Date Time", "Water Level"}.issubset(headers):
                raise ValueError(f"NOAA returned an error or unexpected CSV for {begin}–{end}: "
                                 f"{response.text[:300]}")
            if columns is None:
                columns = headers
            elif columns != headers:
                raise ValueError("NOAA CSV columns changed between chunks")
            count = 0
            lower = datetime.strptime(begin, "%Y%m%d")
            upper = datetime.strptime(end, "%Y%m%d") + timedelta(days=1)
            for raw in reader:
                if None in raw or any(value is None for value in raw.values()):
                    raise ValueError("Malformed NOAA CSV row")
                row = {key.strip(): value.strip() for key, value in raw.items()}
                timestamp = datetime.strptime(row["Date Time"], "%Y-%m-%d %H:%M")
                if not lower <= timestamp < upper:
                    raise ValueError("NOAA returned a timestamp outside its requested chunk")
                if timestamp in rows_by_time and rows_by_time[timestamp] != row:
                    raise ValueError("Conflicting NOAA records for the same timestamp")
                rows_by_time[timestamp] = row
                count += 1
            if not count:
                raise ValueError(f"No observations returned for {begin}–{end}")
    return columns, [rows_by_time[key] for key in sorted(rows_by_time)]


def save_csv(columns, rows, destination):
    """Write only after a successful download; refuse to overwrite an existing file."""
    if not rows:
        raise ValueError("No observations to save")
    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)
    with destination.open("x", newline="", encoding="utf-8") as stream:
        writer = csv.DictWriter(stream, fieldnames=columns)
        writer.writeheader()
        writer.writerows(rows)
    return destination


In [ ]:
# Public example parameters from the supplied notebook; adapt before running.
# MSL = Mean Sea Level. Datum availability depends on the station.
STATION = "8774230"
START_DATE = "20220630"
END_DATE = "20220730"
DATUM = "MSL"

# Relative to the notebook working directory; data/ is excluded from Git.
OUTPUT = Path("data") / (
    f"noaa_{STATION}_water_level_{START_DATE}_{END_DATE}_{DATUM}_metric_gmt.csv"
)


In [ ]:
columns, rows = download_water_levels(STATION, START_DATE, END_DATE, DATUM)
saved_path = save_csv(columns, rows, OUTPUT)
print(f"Saved {len(rows)} observations to {saved_path}")
print("Review missing values, time gaps, and NOAA quality flags before analysis.")
